In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
import joblib

In [3]:
df = pd.read_csv('drom_archive_cleaned_2018-2025.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 16 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   Название машины             1000000 non-null  str    
 1   Год                         1000000 non-null  float64
 2   Дата размещения объявления  1000000 non-null  str    
 3   Цена                        1000000 non-null  float64
 4   Объем двигателя             1000000 non-null  float64
 5   Тип двигателя               1000000 non-null  str    
 6   Мощность                    1000000 non-null  float64
 7   Коробка передач             1000000 non-null  str    
 8   Привод                      1000000 non-null  str    
 9   Пробег                      1000000 non-null  float64
 10  Руль                        1000000 non-null  str    
 11  Поколение                   1000000 non-null  float64
 12  Рестайлинг                  1000000 non-null  float64
 13  Тип кузов

```
Делим дату размещения объявления на год, месяц и возраст, чтобы модели было проще
```

## Линейная регрессия

```
Обозначаем категориальные и числовые признаки
```

In [6]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг', 'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

```
Преобразователь категориальных и числовых признаков
```

In [7]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

```
Модель линейной регрессии с препроцессором
```

In [8]:
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])

```
Разделение данных на целевую переменную и признаки
```

In [9]:
y = df['Цена']
X = df.drop('Цена', axis=1)

```
Разделение на обучающую и тестовую выборки (2018-2023, 2024-2025)
```

In [10]:
X_train = X[X['Год объявления'] <= 2023]
X_test = X[X['Год объявления'] >= 2024]

y_train = y[X['Год объявления'] <= 2023]
y_test = y[X['Год объявления'] >= 2024]
joblib.dump((X_train, X_test, y_train, y_test), "data_split.pkl")
X_train, X_test, y_train, y_test = joblib.load("data_split.pkl")

In [11]:
model.fit(X_train, y_train)
joblib.dump(model, "lr_model.pkl")

['lr_model.pkl']

In [12]:
y_pred = model.predict(X_test)

```
Оценка модели
```

In [13]:
lr_mse = mean_squared_error(y_test, y_pred)
lr_rmse = np.sqrt(lr_mse)
lr_mae = mean_absolute_error(y_test, y_pred)
lr_r2 = r2_score(y_test, y_pred)

In [14]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [lr_mse, lr_rmse, lr_mae, lr_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),397_929_774_235.88
1,Среднеквадратическая ошибка (RMSE),630_816.75
2,Средняя абсолютная ошибка (MAE),328_809.37
3,Коэффицент детерминации (R^2),0.62
